In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from qdrant_client import QdrantClient
from transformers import BitsAndBytesConfig

# --- CONFIG CLASSES ---
@dataclass
class ChunkingConfig:
    chunk_size: int = 512
    chunk_overlap: int = 100
    use_hierarchical: bool = False
    child_chunk_size: int = 128

@dataclass
class RetrievalConfig:
    top_k: int = 5          # Final k documents for LLM
    overfetch_k: int = 15   # Candidates fetched from DB (The "Funnel" top)
    mode: str = "dense"     # "dense" | "hybrid"
    use_reranker: bool = False
    rerank_top_n: int = 5   # Documents to keep after reranking

@dataclass
class LLMConfig:
    model_name: str = "Qwen/Qwen3-4B-Instruct-2507"
    max_new_tokens: int = 256
    context_window: int = 2048
    load_in_4bit: bool = True
    # Subsample for E2E evaluation speed
    e2e_eval_n: int = 10
    prompt_template: str = "Context:\n{context_str}\n\nQuery: {query_str}\nAnswer:"

@dataclass
class EmbeddingConfig:
    model_name: str = "Qwen/Qwen3-Embedding-0.6B"
    truncate_dim: Optional[int] = None # For Matryoshka learning

@dataclass
class RAGConfig:
    name: str = "baseline"
    chunking: ChunkingConfig = field(default_factory=ChunkingConfig)
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    llm: LLMConfig = field(default_factory=LLMConfig)
    embedding: EmbeddingConfig = field(default_factory=EmbeddingConfig)
    qdrant_collection: str = "scifact_dense_base"
    recreate_collection: bool = False
    rerank_model: str = "Qwen/Qwen3-Reranker-0.6B"

# --- MODEL HELPERS ---
_CACHED_LLM = None
_CACHED_EMBED = None

def unload_embedder():
    global _CACHED_EMBED
    if _CACHED_EMBED:
        del _CACHED_EMBED
        _CACHED_EMBED = None
        Settings.embed_model = None
        gc.collect(); torch.cuda.empty_cache()

def get_embedder(cfg: EmbeddingConfig):
    global _CACHED_EMBED
    if _CACHED_EMBED: return _CACHED_EMBED
    _CACHED_EMBED = HuggingFaceEmbedding(
        model_name=cfg.model_name,
        device="cuda",
        normalize=True,
        truncate_dim=cfg.truncate_dim
    )
    return _CACHED_EMBED

def get_llm(cfg: LLMConfig):
    global _CACHED_LLM
    if _CACHED_LLM: return _CACHED_LLM
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    _CACHED_LLM = HuggingFaceLLM(
        model_name=cfg.model_name,
        tokenizer_name=cfg.model_name,
        context_window=cfg.context_window,
        max_new_tokens=cfg.max_new_tokens,
        model_kwargs={"quantization_config": bnb},
        generate_kwargs={"do_sample": False},
        device_map="auto"
    )
    return _CACHED_LLM

In [ ]:
import ir_measures
from ir_measures import nDCG, Recall, RR

RUNS_CSV_PATH = os.path.join(LOGS_DIR, "runs.csv")

def compute_redundancy(contexts: List[str]) -> float:
    """Calculates semantic redundancy (higher = more duplicate info)."""
    if not contexts: return 0.0
    unique = len(set(contexts))
    return 1.0 - (unique / len(contexts))

def compute_retrieval_metrics(qrels_map, run_doc_ids, run_contexts, run_scores, k=5):
    # 1. Reference-based Metrics (Ground Truth needed)
    run_ir = {}
    for qid, doc_ids in run_doc_ids.items():
        run_ir[qid] = {doc_id: float(-(rank + 1)) for rank, doc_id in enumerate(doc_ids[:k])}

    measures = [nDCG@k, Recall@k]
    agg = ir_measures.calc_aggregate(measures, qrels_map, run_ir)

    # 2. Reference-free Metrics (No Ground Truth)
    red_vals = [compute_redundancy(ctxs[:k]) for ctxs in run_contexts.values()]
    # Mean Score: How confident is the retriever?
    score_vals = [np.mean(scores[:k]) for scores in run_scores.values() if scores]

    return {
        "ndcg@k": float(agg[nDCG@k]),
        "recall@k": float(agg[Recall@k]),
        "ref_free_mean_score": float(np.mean(score_vals)) if score_vals else 0.0,
        "ref_free_redundancy": float(np.mean(red_vals)) if red_vals else 0.0
    }

def log_run(cfg, split, stage, run_name, metrics):
    row = {
        "ts": time.strftime("%H:%M:%S", time.gmtime()),
        "run_name": run_name,
        "split": split,
        "stage": stage,
        **metrics
    }
    df_old = pd.read_csv(RUNS_CSV_PATH) if os.path.exists(RUNS_CSV_PATH) else pd.DataFrame()
    pd.concat([df_old, pd.DataFrame([row])], ignore_index=True).to_csv(RUNS_CSV_PATH, index=False)

def show_leaderboard(stage="retrieval", split="main"):
    if not os.path.exists(RUNS_CSV_PATH): return
    df = pd.read_csv(RUNS_CSV_PATH)
    df_sub = df[(df["stage"]==stage) & (df["split"]==split)].copy()
    if df_sub.empty: return

    # Sort by key metric
    sort_col = "ndcg@k" if stage == "retrieval" else "faithfulness"
    if sort_col in df_sub.columns:
        df_sub = df_sub.sort_values(sort_col, ascending=False)

    # Select columns to display
    cols_ret = ["run_name", "ndcg@k", "recall@k", "ref_free_mean_score", "ref_free_redundancy", "latency_s"]
    cols_gen = ["run_name", "faithfulness", "answer_relevancy", "latency_s"]

    cols = cols_ret if stage == "retrieval" else cols_gen
    print(f"\n🏆 LEADERBOARD [{stage.upper()} | {split.upper()}] 🏆")
    display(df_sub[[c for c in cols if c in df_sub.columns]])

def _get_df(split): return df_queries[df_queries["query_id"].isin(split_main if split=="main" else split_challenge)]

def run_retrieval(cfg: RAGConfig, split: str, run_name: str):
    unload_embedder()
    index, retriever, _ = build_rag_pipeline(cfg, df_corpus, load_llm=True)

    df_q = _get_df(split)
    results_dids, results_txts, results_scores = {}, {}, {}

    t0 = time.time()
    for row in tqdm(df_q.itertuples(index=False), total=len(df_q), desc=f"Retr {run_name}"):
        nodes = retriever.retrieve(row.text)
        # De-duplicate by doc_id to avoid metrics skew
        seen, dids, txts, scores = set(), [], [], []
        for n in nodes:
            if n.metadata["doc_id"] not in seen:
                seen.add(n.metadata["doc_id"])
                dids.append(n.metadata["doc_id"])
                txts.append(n.get_text())
                scores.append(n.score if n.score else 0.0)
        results_dids[str(row.query_id)] = dids
        results_txts[str(row.query_id)] = txts
        results_scores[str(row.query_id)] = scores

    # FIX: filter qrels_map based on current split
    split_qids = set(df_q["query_id"].astype(str))
    qrels_map_filtered = {qid: docs for qid, docs in qrels_map.items() if qid in split_qids}
    
    latency = time.time() - t0

    metrics = compute_retrieval_metrics(qrels_map_filtered, results_dids, results_txts, results_scores, k=cfg.retrieval.top_k)
    metrics["latency_s"] = latency
    log_run(cfg, split, "retrieval", run_name, metrics)
    show_leaderboard("retrieval", split)
    unload_embedder()

def run_generation(cfg: RAGConfig, split: str, run_name: str):
    """Generates answers and saves them to JSON for subsequent evaluation."""
    # 1. Retrieval Phase
    unload_embedder()
    index, retriever, _ = build_rag_pipeline(cfg, df_corpus, load_llm=True)

    df_q = _get_df(split).head(cfg.llm.e2e_eval_n)
    records = []
    for row in tqdm(df_q.itertuples(index=False), total=len(df_q), desc="Retrieving Contexts"):
        nodes = retriever.retrieve(row.text)
        ctx_list = [n.get_text() for n in nodes[:cfg.retrieval.top_k]]
        records.append({"query_id": str(row.query_id), "q": row.text, "ctx": "\n".join(ctx_list)})

    unload_embedder()

    # 2. Generation Phase
    Settings.embed_model = None
    llm = get_llm(cfg.llm)

    t0 = time.time()
    for item in tqdm(records, desc="Generating"):
        prompt = cfg.llm.prompt_template.format(context_str=item["ctx"], query_str=item["q"])
        try: 
            item["a"] = llm.complete(prompt).text
        except: 
            item["a"] = ""
    latency = time.time() - t0

    # 3. Save results
    output_path = os.path.join(LOGS_DIR, f"generations_{run_name}_{split}.json")
    import json
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump({
            "records": records, 
            "latency_s": latency, 
            "cfg_name": cfg.name,
            "run_name": run_name,
            "split": split
        }, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Generations saved to {output_path}")
    return output_path


def run_evaluation(generation_file: str, eval_name: Optional[str] = None):
    """Evaluates already generated answers."""
    import json
    
    # Load generated data
    with open(generation_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    records = data["records"]
    latency = data["latency_s"]
    split = data.get("split", "main")
    run_name = eval_name if eval_name else data.get("run_name", "eval")
    
    # Load LLM for Judge
    Settings.embed_model = None
    llm = get_llm(LLMConfig())
    
    # Judge Phase
    faith_scores = []
    rel_scores = []

    for item in tqdm(records, desc=f"Judging {run_name}"):
        if not item.get("a", ""):
            faith_scores.append(0.0); rel_scores.append(0.0)
            continue

        # Metric 1: Faithfulness (Context vs Answer)
        p_faith = f"Context: {item['ctx'][:1000]}\nAnswer: {item['a']}\nDoes the Answer use the Context? YES/NO."
        res_f = llm.complete(p_faith, max_new_tokens=5).text.upper()
        faith_scores.append(1.0 if "YES" in res_f else 0.0)

        # Metric 2: Answer Relevancy (Question vs Answer)
        p_rel = f"Question: {item['q']}\nAnswer: {item['a']}\nDoes the Answer address the Question? YES/NO."
        res_r = llm.complete(p_rel, max_new_tokens=5).text.upper()
        rel_scores.append(1.0 if "YES" in res_r else 0.0)

    metrics = {
        "faithfulness": np.mean(faith_scores),
        "answer_relevancy": np.mean(rel_scores),
        "latency_s": latency
    }
    
    log_run(RAGConfig(name=run_name), split, "e2e", run_name, metrics)
    show_leaderboard("e2e", split)
    return metrics


def run_e2e(cfg: RAGConfig, split: str, run_name: str, generation_file: Optional[str] = None):
    """
    End-to-end evaluation with option to reuse generations.
    
    Args:
        cfg: RAG configuration
        split: "main" or "challenge"
        run_name: Experiment name
        generation_file: Path to previously generated answers file (optional)
    """
    if generation_file and os.path.exists(generation_file):
        # Reuse existing generation
        print(f"📂 Loading generations from {generation_file}")
        return run_evaluation(generation_file, run_name)
    else:
        # Full cycle: generation + evaluation
        gen_file = run_generation(cfg, split, run_name)
        return run_evaluation(gen_file, run_name)
